In [0]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║          KIRO ETL ENGINE  —  Metadata-Driven Data Pipeline          ║
# ║  Catalog  : demo_catalog                                            ║
# ║  Control  : admin.data_flow_control_header                          ║
# ║             admin.data_flow_l0_detail   (L0 ingestion)             ║
# ║             admin.data_flow_pb_detail   (L1/L2 transform)          ║
# ║  Audit    : admin.audit_log                                         ║
# ║  Lookup   : admin.etl_pipeline_lookup                               ║
# ╚══════════════════════════════════════════════════════════════════════╝
pass

In [0]:
# ── Widgets ────────────────────────────────────────────────────────────────
dbutils.widgets.text("GROUP_ID",          "",    "Pipeline Group ID")
dbutils.widgets.text("TARGET_LOAD_TABLE", "",    "Single table to run (blank = ALL)")
dbutils.widgets.text("ENVIRONMENT",       "dev", "Environment (dev/qa/prod)")
dbutils.widgets.text("LOB",               "",    "Line of Business filter (blank = ALL)")
dbutils.widgets.text("RUN_LAYER",         "",    "Layer override: L0 / L1 / L2 / ALL")

GROUP_ID     = dbutils.widgets.get("GROUP_ID").strip().upper()
TARGET_TABLE = dbutils.widgets.get("TARGET_LOAD_TABLE").strip()
ENV          = dbutils.widgets.get("ENVIRONMENT").strip() or "dev"
LOB_FILTER   = dbutils.widgets.get("LOB").strip().upper()
RUN_LAYER    = dbutils.widgets.get("RUN_LAYER").strip().upper()

# Derive layer from GROUP_ID suffix when RUN_LAYER not supplied
if RUN_LAYER:
    LAYER = RUN_LAYER
elif GROUP_ID.endswith("_L0"):
    LAYER = "L0"
elif GROUP_ID.endswith("_L1"):
    LAYER = "L1"
elif GROUP_ID.endswith("_L2"):
    LAYER = "L2"
else:
    LAYER = "ALL"

if not GROUP_ID:
    raise ValueError("[KIRO] GROUP_ID widget is required but was left empty.")

print("")
print("  KIRO ETL ENGINE — PARAMETER SUMMARY")
print("  " + "─" * 44)
print(f"  Group ID     : {GROUP_ID}")
print(f"  Layer        : {LAYER}")
print(f"  Target Table : {TARGET_TABLE or 'ALL (full layer run)'}")
print(f"  LOB Filter   : {LOB_FILTER or 'ALL'}")
print(f"  Environment  : {ENV}")
print("  " + "─" * 44)
print("")

In [0]:
import os
import traceback
import json
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

# ── Constants ─────────────────────────────────────────────────────────────
CATALOG        = "demo_catalog"
CONTROL_SCHEMA = "admin"
VERSION        = "4.0"

# Attempt to get Databricks run_id for audit correlation
try:
    ctx    = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    RUN_ID = ctx.currentRunId().get() if ctx.currentRunId().isDefined() else ""
except Exception:
    RUN_ID = ""

print("  KIRO ETL ENGINE — RUNTIME CONFIGURATION")
print("  " + "─" * 44)
print(f"  Catalog        : {CATALOG}")
print(f"  Control Schema : {CONTROL_SCHEMA}")
print(f"  Engine Version : {VERSION}")
print(f"  Run ID         : {RUN_ID or '(interactive)'}")
print("  " + "─" * 44)
print("")

In [0]:
# ── Structured logger — all output flows through these helpers ─────────────
# Style: clean, left-aligned, consistent indentation, no emoji clutter

def log_header(title):
    bar = "=" * 70
    print(f"\n{bar}")
    print(f"  {title}")
    print(f"{bar}")

def log_section(title):
    print(f"\n  [ {title} ]")
    print("  " + "-" * 50)

def log_info(label, value=""):
    if value:
        print(f"  {label:<22}: {value}")
    else:
        print(f"  {label}")

def log_step(step):
    print(f"  >> {step}")

def log_ok(message):
    print(f"  OK  {message}")

def log_warn(message):
    print(f"  WARN  {message}")

def log_fail(message):
    print(f"  FAIL  {message}")

def log_result(label, value):
    print(f"  --> {label}: {value}")

def log_divider():
    print("  " + "-" * 60)

def log_object_header(idx, total, layer, full_name, obj_type, load_type, priority=None):
    print(f"")
    print(f"  OBJECT [{idx}/{total}]")
    print(f"  " + "-" * 60)
    print(f"  {'Layer':<20}: {layer}")
    print(f"  {'Target':<20}: {full_name}")
    print(f"  {'Object Type':<20}: {obj_type}")
    print(f"  {'Load Strategy':<20}: {load_type}")
    if priority is not None:
        print(f"  {'Priority':<20}: {priority}")

def log_object_footer(status, rows, duration_s):
    print(f"  " + "-" * 60)
    print(f"  {'Status':<20}: {status}")
    print(f"  {'Rows Written':<20}: {rows:,}")
    print(f"  {'Duration':<20}: {duration_s}s")

print("  Logger helpers loaded.")

In [0]:
def read_source(url, fmt="csv", delimiter=","):
    """
    Reads data from HTTP/GitHub, S3, DBFS, or Databricks Volumes.
    Supported formats: csv, tsv, json, parquet, delta, avro, orc, xlsx, xml

    Returns:
        Spark DataFrame

    Raises:
        ValueError  — empty URL or unsupported format
        IOError     — HTTP error or read failure
    """
    fmt = (fmt or "csv").strip().lower()
    url = (url or "").strip()

    if not url:
        raise ValueError("[read_source] SOURCE url/path is empty.")

    log_step(f"read_source  format={fmt}  url={url[:80]}{'...' if len(url) > 80 else ''}")

    # ── HTTP / GitHub ──────────────────────────────────────────────────
    if url.startswith("http"):
        import requests, io, pandas as pd
        try:
            resp = requests.get(url, timeout=120)
        except Exception as e:
            raise IOError(f"[read_source] HTTP connection failed: {e}")

        if not resp.ok:
            raise IOError(f"[read_source] HTTP {resp.status_code} for {url[:80]}")

        log_info("HTTP response", f"{resp.status_code}  size={len(resp.content):,} bytes")

        if fmt in ("csv", "tsv"):
            sep = "\t" if fmt == "tsv" else delimiter
            pdf = pd.read_csv(io.BytesIO(resp.content), sep=sep, low_memory=False, on_bad_lines="skip")
        elif fmt in ("xlsx", "xls", "excel"):
            pdf = pd.read_excel(io.BytesIO(resp.content))
        elif fmt == "json":
            pdf = pd.read_json(io.BytesIO(resp.content))
        elif fmt == "parquet":
            pdf = pd.read_parquet(io.BytesIO(resp.content))
        else:
            raise ValueError(f"[read_source] Format '{fmt}' not supported for HTTP sources.")

        df = spark.createDataFrame(pdf)

    # ── Spark-native (S3, DBFS, Volumes, Delta) ────────────────────────
    else:
        reader = spark.read
        if fmt in ("csv", "tsv"):
            sep = "\t" if fmt == "tsv" else delimiter
            df  = (reader
                   .option("header", "true")
                   .option("inferSchema", "true")
                   .option("delimiter", sep)
                   .option("multiLine", "true")
                   .option("escape", '"')
                   .csv(url))
        elif fmt == "json":
            df = reader.option("multiLine", "true").json(url)
        elif fmt in ("parquet", "avro", "orc", "delta"):
            df = reader.format(fmt).load(url)
        elif fmt == "xml":
            df = reader.format("xml").option("rowTag", delimiter or "row").load(url)
        else:
            raise ValueError(f"[read_source] Unsupported format: '{fmt}'")

    log_info("Rows read",    f"{df.count():,}")
    log_info("Columns",      str(len(df.columns)))
    log_info("Column names", str(df.columns))
    return df

print("  Helper read_source loaded.")

In [0]:
def write_table(df, catalog, schema, table, load_type="FULL",
                merge_keys=None, partition_cols=None, retention_days=None):
    """
    Writes df to {catalog}.{schema}.{table} using the specified strategy.

    Load strategies:
      FULL        — overwrite + overwriteSchema
      APPEND      — append + mergeSchema
      INCREMENTAL — alias for APPEND
      DELTA       — Delta MERGE upsert (requires merge_keys)
      MERGE       — alias for DELTA
      SCD         — SCD Type 2 (requires merge_keys)
      SCD2        — alias for SCD

    Returns:
        int  — row count of the target table after write

    Raises:
        ValueError  — invalid load_type or missing merge_keys
        RuntimeError — write/merge failure
    """
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

    full_name  = f"{catalog}.{schema}.{table}"
    load_type  = (load_type or "FULL").strip().upper()
    part_cols  = [c.strip() for c in (partition_cols or "").split(",") if c.strip()]

    # Validate partition columns exist in DataFrame
    if part_cols:
        missing = [c for c in part_cols if c not in df.columns]
        if missing:
            raise ValueError(
                f"[write_table] Partition column(s) not in DataFrame: {missing}. "
                f"Available: {list(df.columns)}"
            )

    log_step(f"write_table  target={full_name}  strategy={load_type}")
    if part_cols:
        log_info("Partition cols", str(part_cols))

    # ── FULL / OVERWRITE ──────────────────────────────────────────────
    if load_type in ("FULL", "OVERWRITE"):
        w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if part_cols:
            w = w.partitionBy(*part_cols)
        w.saveAsTable(full_name)

    # ── APPEND / INCREMENTAL ──────────────────────────────────────────
    elif load_type in ("APPEND", "INCREMENTAL"):
        w = df.write.format("delta").mode("append").option("mergeSchema", "true")
        if part_cols:
            w = w.partitionBy(*part_cols)
        w.saveAsTable(full_name)

    # ── DELTA / MERGE (upsert) ────────────────────────────────────────
    elif load_type in ("DELTA", "MERGE"):
        if not merge_keys:
            raise ValueError(f"[write_table] LOAD_TYPE={load_type} requires TARGET_PK or SOURCE_PK.")
        keys = [k.strip() for k in merge_keys.split(",") if k.strip()]

        missing_keys = [k for k in keys if k not in df.columns]
        if missing_keys:
            raise ValueError(
                f"[write_table] Merge key(s) not found in DataFrame: {missing_keys}. "
                f"Available: {list(df.columns)}"
            )

        ts   = datetime.now().strftime("%Y%m%d%H%M%S%f")[:16]
        view = f"_mrg_{table[:28]}_{ts}"
        df.createOrReplaceTempView(view)

        # Create target if absent
        try:
            spark.sql(f"SELECT 1 FROM {full_name} LIMIT 1")
        except Exception:
            init = df.limit(0).write.format("delta").mode("append")
            if part_cols:
                init = init.partitionBy(*part_cols)
            init.saveAsTable(full_name)
            log_info("Table created", full_name)

        merge_cond  = " AND ".join([f"tgt.`{k}` = src.`{k}`" for k in keys])
        update_cols = [c for c in df.columns if c not in keys]
        update_set  = ", ".join([f"tgt.`{c}` = src.`{c}`" for c in update_cols])

        merge_sql = f"""
            MERGE INTO {full_name} AS tgt
            USING {view} AS src
            ON {merge_cond}
            WHEN MATCHED THEN UPDATE SET {update_set}
            WHEN NOT MATCHED THEN INSERT *
        """ if update_cols else f"""
            MERGE INTO {full_name} AS tgt
            USING {view} AS src
            ON {merge_cond}
            WHEN NOT MATCHED THEN INSERT *
        """
        try:
            spark.sql(merge_sql)
        finally:
            try: spark.catalog.dropTempView(view)
            except Exception: pass

    # ── SCD Type 2 ────────────────────────────────────────────────────
    elif load_type in ("SCD", "SCD2"):
        if not merge_keys:
            raise ValueError(f"[write_table] LOAD_TYPE={load_type} requires TARGET_PK.")
        keys = [k.strip() for k in merge_keys.split(",") if k.strip()]

        if "scd_start_date" not in df.columns:
            df = df.withColumn("scd_start_date", F.current_timestamp())
        if "scd_end_date" not in df.columns:
            df = df.withColumn("scd_end_date", F.lit(None).cast(TimestampType()))
        if "is_current" not in df.columns:
            df = df.withColumn("is_current", F.lit(True))

        ts   = datetime.now().strftime("%Y%m%d%H%M%S%f")[:16]
        view = f"_scd_{table[:28]}_{ts}"
        df.createOrReplaceTempView(view)

        try:
            spark.sql(f"SELECT 1 FROM {full_name} LIMIT 1")
        except Exception:
            init = df.limit(0).write.format("delta").mode("append")
            if part_cols:
                init = init.partitionBy(*part_cols)
            init.saveAsTable(full_name)

        merge_cond = " AND ".join([f"tgt.`{k}` = src.`{k}`" for k in keys])
        try:
            spark.sql(f"""
                MERGE INTO {full_name} AS tgt
                USING {view} AS src
                ON {merge_cond} AND tgt.is_current = true
                WHEN MATCHED THEN UPDATE SET
                    tgt.scd_end_date = current_timestamp(),
                    tgt.is_current   = false
                WHEN NOT MATCHED THEN INSERT *
            """)
        finally:
            try: spark.catalog.dropTempView(view)
            except Exception: pass

    else:
        raise ValueError(
            f"[write_table] Unsupported LOAD_TYPE='{load_type}'. "
            f"Valid: FULL, APPEND, INCREMENTAL, DELTA, MERGE, SCD, SCD2"
        )

    # ── Optional liquid clustering ────────────────────────────────────
    # (applied when PARTITION_METHOD = LIQUID_CLUSTER)
    # Handled by caller — write_table only does partitionBy here

    # ── Retention ─────────────────────────────────────────────────────
    if retention_days and int(retention_days) > 0:
        _apply_retention(full_name, int(retention_days))

    count = spark.table(full_name).count()
    log_info("Target row count", f"{count:,}")
    return count


def _apply_retention(full_name, days):
    try:
        cols          = spark.table(full_name).columns
        ts_candidates = ["_etl_load_ts", "LOAD_TS", "load_ts", "INSERTED_TS", "inserted_ts"]
        ts_col        = next((c for c in ts_candidates if c in cols), None)
        if not ts_col:
            log_warn(f"Retention skipped — no timestamp column found in {full_name}")
            return
        cutoff = (datetime.now() - timedelta(days=days)).strftime("%Y-%m-%d")
        spark.sql(f"DELETE FROM {full_name} WHERE `{ts_col}` < '{cutoff}'")
        log_info("Retention applied", f"deleted rows with {ts_col} < {cutoff} ({days} days)")
    except Exception as e:
        log_warn(f"Retention failed (non-fatal): {type(e).__name__}: {str(e)[:100]}")

print("  Helper write_table loaded.")

In [0]:
def execute_generic_script(script_code, custom_params_raw=""):
    """
    Executes SQL or Python from GENERIC_SCRIPTS column.
    CUSTOM_SCRIPT_PARAMS (MAP<STRING,STRING>) injected as ${KEY} replacements.

    Returns:
        str  — result message
    """
    if not (script_code or "").strip():
        return "no script"

    log_step("execute_generic_script")

    # Parse custom params (dict from MAP column or JSON string)
    params = {}
    if custom_params_raw:
        if isinstance(custom_params_raw, dict):
            params = custom_params_raw
        else:
            try:
                params = json.loads(str(custom_params_raw))
            except Exception:
                log_warn("CUSTOM_SCRIPT_PARAMS is not valid JSON — params not injected")

    if params:
        log_info("Script params", str(params))

    script = script_code
    for k, v in params.items():
        script = script.replace(f"${{{k}}}", str(v))

    upper = script.strip().upper()
    if any(upper.startswith(kw) for kw in
           ["SELECT", "INSERT", "UPDATE", "DELETE", "CREATE", "ALTER", "DROP", "MERGE"]):
        result = spark.sql(script)
        if upper.startswith("SELECT"):
            cnt = result.count()
            log_info("Script result", f"{cnt:,} rows returned")
            return f"{cnt} rows"
        log_info("Script result", "SQL executed")
        return "SQL executed"
    else:
        exec(script, {"spark": spark, "dbutils": dbutils, "F": F, "params": params})
        log_info("Script result", "Python executed")
        return "Python executed"

print("  Helper execute_generic_script loaded.")

In [0]:
def write_audit(group_id, table_name, layer, status, message, rows,
                start_time, end_time, lob="UNKNOWN"):
    """
    Appends one row to admin.audit_log.
    Uses DataFrame API — safe against quotes/newlines in message.
    Never raises — audit failure must not kill the ETL job.
    """
    try:
        from pyspark.sql import Row
        from pyspark.sql.types import (
            StructType, StructField,
            StringType, TimestampType, LongType, DoubleType
        )

        schema = StructType([
            StructField("DATA_FLOW_GROUP_ID", StringType(),    True),
            StructField("TARGET_TABLE",       StringType(),    True),
            StructField("STATUS",             StringType(),    True),
            StructField("MESSAGE",            StringType(),    True),
            StructField("CREATED_DATE",       TimestampType(), True),
            StructField("ETL_LAYER",          StringType(),    True),
            StructField("LOB",                StringType(),    True),
            StructField("ROWS_PROCESSED",     LongType(),      True),
            StructField("DURATION_SECONDS",   DoubleType(),    True),
            StructField("START_TIME",         TimestampType(), True),
            StructField("END_TIME",           TimestampType(), True),
            StructField("LOAD_TS",            TimestampType(), True),
            StructField("RUN_ID",             StringType(),    True),
            StructField("ENVIRONMENT",        StringType(),    True),
        ])

        now      = datetime.now()
        duration = (end_time - start_time).total_seconds()
        row      = Row(
            DATA_FLOW_GROUP_ID = str(group_id   or ""),
            TARGET_TABLE       = str(table_name or ""),
            STATUS             = str(status     or ""),
            MESSAGE            = str(message    or "")[:400],
            CREATED_DATE       = now,
            ETL_LAYER          = str(layer or ""),
            LOB                = str(lob   or "UNKNOWN"),
            ROWS_PROCESSED     = int(rows  or 0),
            DURATION_SECONDS   = float(duration),
            START_TIME         = start_time,
            END_TIME           = end_time,
            LOAD_TS            = now,
            RUN_ID             = str(RUN_ID or ""),
            ENVIRONMENT        = str(ENV    or "dev"),
        )

        tbl = f"{CATALOG}.{CONTROL_SCHEMA}.audit_log"
        (spark
         .createDataFrame([row], schema=schema)
         .write.format("delta").mode("append")
         .saveAsTable(tbl))

    except Exception as e:
        log_warn(f"write_audit failed (non-fatal): {type(e).__name__}: {str(e)[:150]}")

print("  Helper write_audit loaded.")

In [0]:
def write_lookup(group_id, layer, lob, source_ref, target_full_name,
                 load_type, obj_type, rows, status, duration_s,
                 merge_keys="", partition_cols="", run_ts=None):
    """
    Upserts one row into admin.etl_pipeline_lookup after each object completes.
    Key: (DATA_FLOW_GROUP_ID, ETL_LAYER, TARGET_FULL_NAME)

    This table gives a live view of:
      - which pipelines and tables exist
      - last run status, row count, duration
      - source/target lineage
      - key columns and load strategy

    Never raises — lookup failure must not kill the ETL job.
    """
    try:
        from pyspark.sql import Row
        from pyspark.sql.types import (
            StructType, StructField,
            StringType, TimestampType, LongType, DoubleType
        )

        schema = StructType([
            StructField("DATA_FLOW_GROUP_ID",  StringType(),    True),
            StructField("ETL_LAYER",           StringType(),    True),
            StructField("LOB",                 StringType(),    True),
            StructField("SOURCE_REFERENCE",    StringType(),    True),
            StructField("TARGET_FULL_NAME",    StringType(),    True),
            StructField("LOAD_TYPE",           StringType(),    True),
            StructField("OBJECT_TYPE",         StringType(),    True),
            StructField("MERGE_KEYS",          StringType(),    True),
            StructField("PARTITION_COLS",      StringType(),    True),
            StructField("LAST_STATUS",         StringType(),    True),
            StructField("LAST_ROW_COUNT",      LongType(),      True),
            StructField("LAST_DURATION_SECS",  DoubleType(),    True),
            StructField("LAST_RUN_TS",         TimestampType(), True),
            StructField("RUN_ID",              StringType(),    True),
            StructField("ENVIRONMENT",         StringType(),    True),
        ])

        now = run_ts or datetime.now()
        row = Row(
            DATA_FLOW_GROUP_ID = str(group_id          or ""),
            ETL_LAYER          = str(layer              or ""),
            LOB                = str(lob                or "UNKNOWN"),
            SOURCE_REFERENCE   = str(source_ref         or "")[:500],
            TARGET_FULL_NAME   = str(target_full_name   or ""),
            LOAD_TYPE          = str(load_type          or ""),
            OBJECT_TYPE        = str(obj_type           or "TABLE"),
            MERGE_KEYS         = str(merge_keys         or ""),
            PARTITION_COLS     = str(partition_cols     or ""),
            LAST_STATUS        = str(status             or ""),
            LAST_ROW_COUNT     = int(rows               or 0),
            LAST_DURATION_SECS = float(duration_s       or 0.0),
            LAST_RUN_TS        = now,
            RUN_ID             = str(RUN_ID             or ""),
            ENVIRONMENT        = str(ENV                or "dev"),
        )

        lkp_tbl = f"{CATALOG}.{CONTROL_SCHEMA}.etl_pipeline_lookup"
        ts_str  = now.strftime("%Y%m%d%H%M%S%f")[:16]
        view    = f"_lkp_{str(target_full_name).replace('.','_')[:30]}_{ts_str}"

        spark.createDataFrame([row], schema=schema).createOrReplaceTempView(view)

        # Ensure table exists
        try:
            spark.sql(f"SELECT 1 FROM {lkp_tbl} LIMIT 1")
        except Exception:
            (spark.createDataFrame([row], schema=schema)
             .limit(0).write.format("delta").mode("append")
             .saveAsTable(lkp_tbl))

        spark.sql(f"""
            MERGE INTO {lkp_tbl} AS tgt
            USING {view} AS src
            ON  tgt.DATA_FLOW_GROUP_ID = src.DATA_FLOW_GROUP_ID
            AND tgt.ETL_LAYER          = src.ETL_LAYER
            AND tgt.TARGET_FULL_NAME   = src.TARGET_FULL_NAME
            WHEN MATCHED THEN UPDATE SET
                tgt.LAST_STATUS        = src.LAST_STATUS,
                tgt.LAST_ROW_COUNT     = src.LAST_ROW_COUNT,
                tgt.LAST_DURATION_SECS = src.LAST_DURATION_SECS,
                tgt.LAST_RUN_TS        = src.LAST_RUN_TS,
                tgt.RUN_ID             = src.RUN_ID,
                tgt.LOAD_TYPE          = src.LOAD_TYPE,
                tgt.SOURCE_REFERENCE   = src.SOURCE_REFERENCE,
                tgt.MERGE_KEYS         = src.MERGE_KEYS,
                tgt.PARTITION_COLS     = src.PARTITION_COLS
            WHEN NOT MATCHED THEN INSERT *
        """)
        try: spark.catalog.dropTempView(view)
        except Exception: pass

    except Exception as e:
        log_warn(f"write_lookup failed (non-fatal): {type(e).__name__}: {str(e)[:150]}")

print("  Helper write_lookup loaded.")

In [0]:
def process_layer(detail_table, layer):
    """
    Processes all active objects for one ETL layer.

    L0  reads from: data_flow_l0_detail
         columns  : SOURCE, SOURCE_OBJ_SCHEMA, SOURCE_OBJ_NAME,
                    INPUT_FILE_FORMAT, DELIMETER, DQ_LOGIC, CDC_LOGIC,
                    TRANSFORM_QUERY (MAP<STRING,STRING>), PARTITION

    L1/L2 reads from: data_flow_pb_detail
           columns  : TARGET_OBJ_SCHEMA, TARGET_OBJ_NAME, PRIORITY,
                      TARGET_OBJ_TYPE, TRANSFORM_QUERY (STRING),
                      SOURCE_PK, TARGET_PK, PARTITION_OR_INDEX,
                      PARTITION_METHOD, RETENTION_DETAILS, CUSTOM_SCRIPT_PARAMS

    Returns True if all objects succeeded.
    Raises RuntimeError if any object failed.
    """
    if layer not in ("L0", "L1", "L2"):
        raise ValueError(f"[process_layer] Invalid layer '{layer}'. Must be L0, L1 or L2.")

    log_header(f"KIRO ETL ENGINE  |  Layer: {layer}  |  Group: {GROUP_ID}")
    log_info("Control Table",  f"{CATALOG}.{CONTROL_SCHEMA}.{detail_table}")
    log_info("Target Filter",  TARGET_TABLE or "ALL")
    log_info("LOB Filter",     LOB_FILTER   or "ALL")
    log_info("Environment",    ENV)

    # ── Build WHERE filters ────────────────────────────────────────────
    obj_col = "SOURCE_OBJ_NAME" if layer == "L0" else "TARGET_OBJ_NAME"

    filters = [
        f"DATA_FLOW_GROUP_ID = '{GROUP_ID}'",
        "IS_ACTIVE = 'Y'"
    ]
    if TARGET_TABLE and TARGET_TABLE.upper() != "ALL":
        filters.append(f"{obj_col} = '{TARGET_TABLE}'")
    if LOB_FILTER and LOB_FILTER.upper() != "ALL":
        filters.append(f"LOB = '{LOB_FILTER}'")
    where_clause = " AND ".join(filters)

    # ── Layer-specific SELECT (columns never mixed across tables) ──────
    if layer == "L0":
        query = f"""
            SELECT
                DATA_FLOW_GROUP_ID,
                LOB,
                SOURCE,
                SOURCE_OBJ_SCHEMA,
                SOURCE_OBJ_NAME,
                LOAD_TYPE,
                INPUT_FILE_FORMAT,
                STORAGE_TYPE,
                DQ_LOGIC,
                DELIMETER,
                CUSTOM_SCHEMA,
                CDC_LOGIC,
                TRANSFORM_QUERY,
                PRESTAG_FLAG,
                `PARTITION`  AS PARTITION_COLS,
                LS_FLAG,
                LS_DETAIL,
                IS_ACTIVE,
                DEPLOYMENT_SOURCE_DFG
            FROM {CATALOG}.{CONTROL_SCHEMA}.{detail_table}
            WHERE {where_clause}
            ORDER BY SOURCE_OBJ_NAME
        """
    else:
        query = f"""
            SELECT
                DATA_FLOW_GROUP_ID,
                LOB,
                SOURCE,
                TARGET_OBJ_SCHEMA,
                TARGET_OBJ_NAME,
                PRIORITY,
                TARGET_OBJ_TYPE,
                TRANSFORM_QUERY,
                GENERIC_SCRIPTS,
                SOURCE_PK,
                TARGET_PK,
                LOAD_TYPE,
                IS_ACTIVE,
                LS_FLAG,
                LS_DETAIL,
                PARTITION_OR_INDEX,
                PARTITION_METHOD,
                CUSTOM_SCRIPT_PARAMS,
                RETENTION_DETAILS,
                DEPLOYMENT_SOURCE_DFG
            FROM {CATALOG}.{CONTROL_SCHEMA}.{detail_table}
            WHERE {where_clause}
            ORDER BY COALESCE(PRIORITY, 999), TARGET_OBJ_NAME
        """

    log_section("Querying Metadata")
    log_step(f"SELECT from {detail_table} WHERE {where_clause}")

    try:
        rows = spark.sql(query).collect()
    except Exception as e:
        raise RuntimeError(
            f"[process_layer] Metadata query failed on {detail_table}.\n"
            f"  WHERE: {where_clause}\n"
            f"  Error: {type(e).__name__}: {str(e)[:300]}"
        ) from e

    if not rows:
        log_warn(f"No active rows found in {detail_table} for GROUP_ID='{GROUP_ID}'. Layer {layer} skipped.")
        return True

    log_ok(f"Metadata loaded — {len(rows)} object(s) to process")

    # ── Process each object ────────────────────────────────────────────
    failed_objects = []
    success_count  = 0

    for idx, row in enumerate(rows, 1):
        r            = row.asDict()
        t0           = datetime.now()
        status       = "FAILED"
        msg          = ""
        count        = 0
        target_table = None
        source_ref   = ""
        merge_keys   = ""
        partition_cols = ""
        obj_type     = "TABLE"
        lob          = ""
        full_name    = ""

        try:
            lob      = (r.get("LOB")      or "").strip()
            load_type= (r.get("LOAD_TYPE")or "FULL").strip().upper()
            ls_flag  = (r.get("LS_FLAG")  or "N").strip().upper()
            ls_detail= (r.get("LS_DETAIL")or "").strip()

            # ══════════════════════════════════════════════════════════
            # L0 — FILE INGESTION
            # ══════════════════════════════════════════════════════════
            if layer == "L0":
                source_url     = (r.get("SOURCE")           or "").strip()
                target_schema  = (r.get("SOURCE_OBJ_SCHEMA")or "").strip()
                target_table   = (r.get("SOURCE_OBJ_NAME")  or "").strip()
                file_format    = (r.get("INPUT_FILE_FORMAT")or "csv").strip().lower()
                delimiter      = (r.get("DELIMETER")        or ",").strip()
                dq_logic       = (r.get("DQ_LOGIC")         or "").strip()
                cdc_logic      = (r.get("CDC_LOGIC")        or "").strip()
                cast_map       =  r.get("TRANSFORM_QUERY")
                partition_cols = (r.get("PARTITION_COLS")   or "").strip()
                obj_type       = "TABLE"

                if not source_url or not target_schema or not target_table:
                    raise ValueError(
                        f"Missing required L0 metadata: "
                        f"SOURCE='{source_url}' "
                        f"SOURCE_OBJ_SCHEMA='{target_schema}' "
                        f"SOURCE_OBJ_NAME='{target_table}'"
                    )

                resolved_url   = ls_detail if (ls_flag == "Y" and ls_detail) else source_url
                target_table   = os.path.splitext(target_table)[0]
                full_name      = f"{CATALOG}.{target_schema}.{target_table}"
                source_ref     = resolved_url

                log_object_header(idx, len(rows), layer, full_name, obj_type, load_type)
                log_info("LOB",            lob or "(not set)")
                log_info("Source",         resolved_url[:100])
                log_info("Format",         file_format)
                log_info("Delimiter",      repr(delimiter))
                if partition_cols:
                    log_info("Partition",  partition_cols)
                if dq_logic:
                    log_info("DQ Logic",   dq_logic)
                if cdc_logic:
                    log_info("CDC Logic",  cdc_logic)
                if ls_flag == "Y":
                    log_info("LS Override", "source replaced by LS_DETAIL")

                # Read
                log_section("Reading Source")
                df        = read_source(resolved_url, file_format, delimiter)
                raw_count = df.count()

                # Cast MAP
                if cast_map and isinstance(cast_map, dict) and cast_map:
                    log_section("Applying Cast Map")
                    log_info("Columns to cast", str(len(cast_map)))
                    for col_name, cast_expr in cast_map.items():
                        if col_name in df.columns:
                            df = df.withColumn(col_name, F.expr(cast_expr))
                            log_info(f"  cast", f"{col_name}  ->  {cast_expr}")
                        else:
                            log_warn(f"Cast column '{col_name}' not in DataFrame — skipped")

                # DQ filter
                if dq_logic:
                    log_section("Applying DQ Filter")
                    before_dq = df.count()
                    df        = df.filter(F.expr(dq_logic))
                    after_dq  = df.count()
                    log_info("Rows before DQ", f"{before_dq:,}")
                    log_info("Rows after DQ",  f"{after_dq:,}")
                    log_info("Rows dropped",   f"{before_dq - after_dq:,}")

                # ETL audit columns
                df = (df
                      .withColumn("_etl_group_id", F.lit(GROUP_ID))
                      .withColumn("_etl_layer",    F.lit(layer))
                      .withColumn("_etl_lob",      F.lit(lob))
                      .withColumn("_etl_env",      F.lit(ENV))
                      .withColumn("_etl_load_ts",  F.current_timestamp()))

                # Write
                log_section("Writing to Delta")
                count = write_table(df, CATALOG, target_schema, target_table,
                                    load_type=load_type,
                                    partition_cols=partition_cols)
                status = "SUCCESS"
                msg    = f"{count:,} rows loaded"
                success_count += 1

            # ══════════════════════════════════════════════════════════
            # L1/L2 — TRANSFORMATION
            # ══════════════════════════════════════════════════════════
            else:
                target_schema    = (r.get("TARGET_OBJ_SCHEMA")    or "").strip()
                target_table     = (r.get("TARGET_OBJ_NAME")      or "").strip()
                obj_type         = (r.get("TARGET_OBJ_TYPE")      or "Table").strip().upper()
                transform_query  = (r.get("TRANSFORM_QUERY")      or "").strip()
                generic_scripts  = (r.get("GENERIC_SCRIPTS")      or "").strip()
                merge_keys       = (r.get("TARGET_PK") or r.get("SOURCE_PK") or "").strip()
                partition_cols   = (r.get("PARTITION_OR_INDEX")   or "").strip()
                partition_method = (r.get("PARTITION_METHOD")     or "").strip()
                retention_str    = (r.get("RETENTION_DETAILS")    or "").strip()
                custom_params_raw=  r.get("CUSTOM_SCRIPT_PARAMS")
                priority         =  r.get("PRIORITY") or 999
                source_schema    = (r.get("SOURCE")               or "").strip()

                retention_days = None
                if retention_str:
                    try:
                        retention_days = int(retention_str)
                    except Exception:
                        log_warn(f"RETENTION_DETAILS='{retention_str}' is not a valid integer — retention skipped")

                if not target_schema or not target_table:
                    raise ValueError(
                        f"Missing required L1/L2 metadata: "
                        f"TARGET_OBJ_SCHEMA='{target_schema}' "
                        f"TARGET_OBJ_NAME='{target_table}'"
                    )
                if not transform_query:
                    raise ValueError(
                        f"TRANSFORM_QUERY is empty for {target_schema}.{target_table}. "
                        f"Provide a full SQL SELECT in data_flow_pb_detail."
                    )

                full_name  = f"{CATALOG}.{target_schema}.{target_table}"
                source_ref = source_schema

                log_object_header(idx, len(rows), layer, full_name, obj_type, load_type, priority)
                log_info("LOB",           lob or "(not set)")
                log_info("Source Schema", source_schema or "(none)")
                if merge_keys:
                    log_info("Merge Keys",   merge_keys)
                if partition_cols:
                    log_info("Partition",    f"{partition_cols}  [{partition_method or 'PARTITION'}]")
                if retention_days:
                    log_info("Retention",    f"{retention_days} days")
                log_info("Transform SQL", transform_query[:150].replace("\n", " ") +
                                          ("..." if len(transform_query) > 150 else ""))

                # Pre generic script
                if ls_flag == "B" and generic_scripts:
                    log_section("Pre-Script Execution")
                    execute_generic_script(generic_scripts, custom_params_raw)

                # MV handling
                if obj_type == "MV":
                    log_section("Creating Materialized View")
                    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{target_schema}")
                    try:
                        spark.sql(f"CREATE OR REPLACE MATERIALIZED VIEW {full_name} AS {transform_query}")
                        status = "SUCCESS"
                        msg    = "Materialized view created/refreshed"
                        count  = 0
                        log_ok(f"MV ready: {full_name}")
                    except Exception as mv_err:
                        if any(k in str(mv_err).lower() for k in ["materialized view", "not supported", "dlt"]):
                            log_warn("MV not supported in Serverless Job context (Free Edition)")
                            log_step(f"Fallback: creating regular TABLE at {full_name}")
                            df    = spark.sql(transform_query)
                            count = write_table(df, CATALOG, target_schema, target_table, load_type="FULL")
                            status = "SUCCESS"
                            msg    = f"{count:,} rows written as TABLE (MV fallback)"
                            log_ok(msg)
                        else:
                            raise
                    success_count += 1

                else:
                    # Auto-prefix catalog on bare schema.table references
                    if (source_schema
                            and f"{source_schema}." in transform_query
                            and f"{CATALOG}.{source_schema}." not in transform_query):
                        transform_query = transform_query.replace(
                            f"{source_schema}.", f"{CATALOG}.{source_schema}."
                        )
                        log_info("Catalog prefix", f"auto-added '{CATALOG}' to schema '{source_schema}'")

                    # Execute transform
                    log_section("Executing Transform Query")
                    df    = spark.sql(transform_query)
                    if df is None:
                        raise ValueError("TRANSFORM_QUERY returned None")
                    t_count = df.count()
                    log_info("Rows produced",  f"{t_count:,}")
                    log_info("Columns",        str(len(df.columns)))
                    log_info("Column names",   str(df.columns))

                    # ETL audit columns
                    df = (df
                          .withColumn("_etl_group_id", F.lit(GROUP_ID))
                          .withColumn("_etl_layer",    F.lit(layer))
                          .withColumn("_etl_lob",      F.lit(lob))
                          .withColumn("_etl_env",      F.lit(ENV))
                          .withColumn("_etl_load_ts",  F.current_timestamp()))

                    # Write
                    log_section("Writing to Delta")
                    count = write_table(df, CATALOG, target_schema, target_table,
                                        load_type=load_type,
                                        merge_keys=merge_keys,
                                        partition_cols=partition_cols,
                                        retention_days=retention_days)
                    status = "SUCCESS"
                    msg    = f"{count:,} rows processed"
                    success_count += 1

                    # Post generic script
                    if ls_flag == "A" and generic_scripts:
                        log_section("Post-Script Execution")
                        execute_generic_script(generic_scripts, custom_params_raw)

        except Exception as e:
            status = "FAILED"
            msg    = f"{type(e).__name__}: {str(e)[:300]}"
            log_fail(f"Object: {target_table or 'UNKNOWN'}")
            log_fail(f"Error : {msg}")
            print("")
            print("  TRACEBACK:")
            print("  " + "-" * 60)
            traceback.print_exc()
            print("  " + "-" * 60)
            failed_objects.append({"table": target_table or "UNKNOWN", "error": msg})

        finally:
            t1         = datetime.now()
            duration_s = round((t1 - t0).total_seconds(), 2)
            log_object_footer(status, count, duration_s)
            write_audit(GROUP_ID, target_table or "UNKNOWN", layer, status,
                        msg, count, t0, t1, lob)
            write_lookup(
                group_id       = GROUP_ID,
                layer          = layer,
                lob            = lob,
                source_ref     = source_ref,
                target_full_name = full_name,
                load_type      = load_type if 'load_type' in dir() else "FULL",
                obj_type       = obj_type,
                rows           = count,
                status         = status,
                duration_s     = duration_s,
                merge_keys     = merge_keys,
                partition_cols = partition_cols,
                run_ts         = t1,
            )

    # ── Layer summary ──────────────────────────────────────────────────
    log_header(f"LAYER {layer} COMPLETE  |  {success_count}/{len(rows)} succeeded  |  {len(failed_objects)} failed")

    if failed_objects:
        log_fail("Failed objects:")
        for fo in failed_objects:
            log_fail(f"  {fo['table']}  ->  {fo['error'][:120]}")
        print("")
        raise RuntimeError(
            f"Layer {layer}: {len(failed_objects)} object(s) failed. "
            f"Diagnose with: "
            f"SELECT * FROM {CATALOG}.{CONTROL_SCHEMA}.audit_log "
            f"WHERE DATA_FLOW_GROUP_ID='{GROUP_ID}' ORDER BY LOAD_TS DESC"
        )

    return True

print("  Core process_layer loaded.")

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# KIRO ETL ENGINE — MAIN EXECUTION
# ══════════════════════════════════════════════════════════════════════════

pipeline_start = datetime.now()

print("")
print("=" * 70)
print("  KIRO ETL ENGINE v" + VERSION + "  —  PIPELINE START")
print("  " + "─" * 60)
print(f"  Group ID     : {GROUP_ID}")
print(f"  Layer        : {LAYER}")
print(f"  Target Table : {TARGET_TABLE or 'ALL'}")
print(f"  LOB Filter   : {LOB_FILTER or 'ALL'}")
print(f"  Environment  : {ENV}")
print(f"  Run ID       : {RUN_ID or '(interactive)'}")
print(f"  Started At   : {pipeline_start.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)
print("")

LAYER_TABLE_MAP = {
    "L0": "data_flow_l0_detail",
    "L1": "data_flow_pb_detail",
    "L2": "data_flow_pb_detail",
}

layers_run     = []
pipeline_ok    = True
pipeline_error = None

try:
    if LAYER == "ALL":
        layers_run = ["L0", "L1", "L2"]
        for lyr in layers_run:
            process_layer(LAYER_TABLE_MAP[lyr], lyr)
    elif LAYER in LAYER_TABLE_MAP:
        layers_run = [LAYER]
        process_layer(LAYER_TABLE_MAP[LAYER], LAYER)
    else:
        raise ValueError(
            f"Invalid LAYER='{LAYER}'. Valid values: L0, L1, L2, ALL."
        )

except Exception as exc:
    pipeline_ok    = False
    pipeline_error = exc

# ── Pipeline-level audit entry ─────────────────────────────────────────
pipeline_end      = datetime.now()
pipeline_duration = round((pipeline_end - pipeline_start).total_seconds(), 2)
pipeline_status   = "SUCCESS" if pipeline_ok else "FAILED"
pipeline_msg      = "All layers completed" if pipeline_ok else f"{type(pipeline_error).__name__}: {str(pipeline_error)[:300]}"

write_audit(
    group_id   = GROUP_ID,
    table_name = "PIPELINE",
    layer      = LAYER,
    status     = pipeline_status,
    message    = pipeline_msg,
    rows       = 0,
    start_time = pipeline_start,
    end_time   = pipeline_end,
)

# ── Final output ───────────────────────────────────────────────────────
print("")
print("=" * 70)
print(f"  KIRO ETL ENGINE v{VERSION}  —  PIPELINE {pipeline_status}")
print("  " + "─" * 60)
print(f"  Group ID   : {GROUP_ID}")
print(f"  Layers Run : {', '.join(layers_run) if layers_run else LAYER}")
print(f"  Status     : {pipeline_status}")
print(f"  Duration   : {pipeline_duration}s")
print(f"  Finished At: {pipeline_end.strftime('%Y-%m-%d %H:%M:%S')}")

if pipeline_ok:
    print("")
    print("  Query pipeline lookup:")
    print(f"  SELECT * FROM {CATALOG}.{CONTROL_SCHEMA}.etl_pipeline_lookup")
    print(f"  WHERE DATA_FLOW_GROUP_ID = '{GROUP_ID}'")
    print(f"  ORDER BY ETL_LAYER, TARGET_FULL_NAME")
else:
    print("")
    print(f"  Error : {str(pipeline_error)[:200]}")
    print("")
    print("  Diagnose with:")
    print(f"  SELECT * FROM {CATALOG}.{CONTROL_SCHEMA}.audit_log")
    print(f"  WHERE DATA_FLOW_GROUP_ID = '{GROUP_ID}'")
    print(f"  ORDER BY LOAD_TS DESC")

print("=" * 70)
print("")

# Fail the job cell so Jenkins/Databricks marks the run as failed
if not pipeline_ok:
    raise RuntimeError(
        f"KIRO ETL ENGINE: pipeline failed for GROUP_ID='{GROUP_ID}' LAYER='{LAYER}'. "
        f"Root cause: {pipeline_msg}"
    ) from pipeline_error